In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
_project_root = str(
    next(
        (
            _p
            for _p in (Path.cwd(), *Path.cwd().parents)
            if (_p / "common.py").is_file()
        ),
        Path.cwd().parent,
    )
)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import importlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

import figures_paper.figure_config as figure_config
import figures_paper.baseline_comparison as baseline_comparison
from common import Wildcards, evaluations_dir, test_samples, training_samples

importlib.reload(figure_config)
from figures_paper.figure_config import (
    BARPLOT_REF_LINE_COLORS,
    BARPLOT_REF_LINE_STYLES,
    CONDITION_LABELS,
    FONTSIZE,
    LW_MEDIUM,
    LW_THICK,
    LW_THIN,
    OBS_LABELS,
    REF_DISPLAY_MAP,
    REF_LABELS,
    REF_PALETTE,
    configure_axis_spines,
)
from figures_paper.baseline_comparison import (
    HELDOUT_SPLITS,
    load_scored_measurements,
    matched_combo_rmse,
    noise_floor,
    paired_tests,
    plot_rmse_by_method,
    replicate_pairs,
)
from training_configuration import BEST_FEATURE_SETS


## Configuration

In [ ]:
# Style is applied automatically by figure_config on import.
# Re-apply if needed after reload:
# apply_figure_style()

# ── Model ─────────────────────────────────────────────────────────────
data = "dream_cytof"
figure = "figure1a"

models = {
    "DMM": "EGFR_MAPK__logobs",
}

selected_context = "cytof_init"
selected_features = BEST_FEATURE_SETS[selected_context]
selected_model = "DMM"

# Held-out predictions shown as "test". "test": the five test lines, predicted
# by the models trained on all 58 training lines (training RMSEs from the same
# models). "cv+test": also the five CV-validation lines, each predicted by the
# split that held it out; these lines informed the hyperparameter choice. Fig. 1C
# before revision used "cv+test".
HELDOUT = "test"

# Fig. 1C now shows ordinary least squares as "Linear reg.", so the elastic net
# keeps its own name in the trajectory legends
REF_LABELS = {**REF_LABELS, "elasticnet": "Elastic net"}



## Load trajectory data

In [ ]:
# Load trajectories from all specified models
traj_dfs = []
for label, model_name in models.items():
    path = evaluations_dir / model_name / data / f"trajectories_{figure}.csv"
    if not path.exists():
        print(f"  ⚠ Skipping {label}: {path} not found")
        continue
    df = pd.read_csv(path, index_col=0)
    df["model_label"] = label
    traj_dfs.append(df)
    print(
        f"[{label}] {len(df):,} rows, "
        f"{df['sample'].nunique()} cell lines, "
        f"refs: {sorted(df['ref'].unique())}, "
        f"contexts: {sorted(df['context'].unique())}"
    )

traj = pd.concat(traj_dfs, ignore_index=True)
heldout_splits = HELDOUT_SPLITS[HELDOUT]
if heldout_splits is not None:
    traj = traj[traj["samples"].isin(heldout_splits)]

# Summary
print(f"\nCombined: {len(traj):,} rows")
print(f"  Observables : {sorted(traj['observable'].unique())}")
print(f"  Conditions  : {sorted(traj['condition'].unique())}")
print(f"  Time points : {sorted(traj['time'].unique())}")
print(f"  Cell lines  : {traj['sample'].nunique()}")
print(f"  Datasets    : {sorted(traj['dataset'].unique())}")

# Datapoints per (observable, condition, timepoint, cell-line) combo, by ref
print(f"\nDatapoints per (observable, condition, timepoint, cell-line) combo:")
combo_counts = (
    traj.groupby(["ref", "observable", "condition", "time", "sample"])
    .size()
    .reset_index(name="n")
)
for ref_name, ref_group in combo_counts.groupby("ref"):
    counts = ref_group["n"]
    print(
        f"  ref={ref_name:15s}: "
        f"{len(ref_group):5d} combos, "
        f"n per combo: min={counts.min()}, median={counts.median():.0f}, max={counts.max()}"
    )

# ── Load per-combo RMSE from by_cl_cond_obs CSV ──────────────────────
by_path = evaluations_dir / models[selected_model] / data / f"by_cl_cond_obs_{figure}.csv"
by_cl = pd.read_csv(by_path, index_col=0).rename(columns={"sample": "cell_line"})
if heldout_splits is not None:
    by_cl = by_cl[by_cl["samples"].isin(heldout_splits)]

by_dmm_val = by_cl[
    (by_cl["ref"] == "DMM")
    & (by_cl["context"] == selected_context)
    & (by_cl["dataset"] == "val")
]
combo_rmse = by_dmm_val.groupby(["cell_line", "condition", "observable"])["rmse"].median().sort_values()

# ── Select 3 representative combos at RMSE quartiles ──────────────────
quantiles = [0.05, 0.5, 0.95]
q1, q2, q3 = combo_rmse.quantile(quantiles)
combo_q1 = (combo_rmse - q1).abs().idxmin()
combo_q2 = (combo_rmse - q2).abs().idxmin()
combo_q3 = (combo_rmse - q3).abs().idxmin()

HIGHLIGHT_CELL_LINES = [combo_q1[0], combo_q2[0], combo_q3[0]]
print(f"\nRepresentative combos (Q{quantiles[0]*100:.0f} / Q{quantiles[1]*100:.0f} / Q{quantiles[2]*100:.0f} of median per-seed RMSE on val set):")

rep_combos = {}
for quant, combo in zip(quantiles, [combo_q1, combo_q2, combo_q3]):
    cl, cond, obs = combo
    obs_label = OBS_LABELS.get(obs, obs)
    cond_label = CONDITION_LABELS.get(cond, cond)
    cl_display = cl.replace("c", "", 1)
    tag = f"Q{quant*100:.0f}"
    rep_combos[tag] = combo
    print(f"  {tag}  → {cl_display} / {cond_label} / {obs_label}  (RMSE = {combo_rmse[combo]:.4f})")
print(f"  Range: {combo_rmse.min():.4f} – {combo_rmse.max():.4f} ({len(combo_rmse)} combos)")

## Baselines and measurement noise floor

Per (cell line × condition × observable) RMSE of the DMM and of the reference
models: the mechanistic model with population-average parameters (negative
control), the per-cell-line fits (positive control) and linear, lasso and
elastic-net regression on the DMM's input features. All are scored on the same
individual replicate measurements; the regressors predict one value per time
point and are matched on time point. DMM values are medians over random seeds.

The noise floor is the SD of a single measurement, pooled over replicate pairs
(same cell line, condition, observable and time point): the RMSE that exact
prediction of every expected value would still incur.

In [ ]:
method_rmse, regressor_coverage = matched_combo_rmse(
    traj, selected_context, selected_features
)
print("Fraction of measurements with a regressor prediction:")
display(regressor_coverage)

scored_cell_lines = training_samples(Wildcards(data, "all")) + test_samples
replicates = replicate_pairs(
    load_scored_measurements(models[selected_model], data, scored_cell_lines)
)
replicates["cell_lines"] = np.where(
    replicates["cell_line"].isin(test_samples), "test", "train"
)
noise_sigma = noise_floor(replicates)["sigma"].item()
noise_table = pd.concat(
    [
        noise_floor(replicates),
        noise_floor(replicates, ["pairing"]),
        noise_floor(replicates, ["cell_lines"]),
        noise_floor(replicates, ["pairing", "observable"]),
    ],
    ignore_index=True,
)
noise_table.to_csv("fig1c_noise_floor.csv", index=False)
print(f"Noise floor: {noise_sigma:.3f}")
display(noise_table)

## Helper functions

In [ ]:
def plot_trajectory(
    traj,
    cell_line,
    condition,
    observable,
    model_label=None,
    context=None,
    refs=None,
    ax=None,
    show_seeds=True,
    title=None,
    rmse_data=None,
):
    """
    Plot a single trajectory: experimental data (obs) + model simulations.

    Parameters
    ----------
    traj : DataFrame
        Full trajectory table.
    cell_line, condition, observable : str
        Which trajectory to plot.
    model_label : str, optional
        Filter to a specific model (e.g. "DMM + tEGFR"). None = first available.
    context : str, optional
        Filter to a specific context (e.g. "cytof_init"). None = first available.
    refs : list of str, optional
        Which reference methods to show. None = all.
    ax : matplotlib Axes, optional
    show_seeds : bool
        If True, show individual random-seed trajectories for DMM.
    title : str, optional
        Custom title. Auto-generated if None.
    rmse_data : dict, optional
        RMSE values by ref name to display as annotations.
    """
    sub = traj[
        (traj["sample"] == cell_line)
        & (traj["condition"] == condition)
        & (traj["observable"] == observable)
    ].copy()

    if model_label is not None:
        sub = sub[sub["model_label"] == model_label]
    else:
        model_label = sub["model_label"].iloc[0] if len(sub) else "?"

    if context is not None:
        sub = sub[sub["context"] == context]
    else:
        context = sub["context"].iloc[0] if len(sub) else "?"

    if refs is not None:
        sub = sub[sub["ref"].isin(refs)]

    if sub.empty:
        print(f"No data for {cell_line} / {condition} / {observable}")
        return ax

    if ax is None:
        _, ax = plt.subplots(figsize=(5, 3.5))

    # --- Experimental data from obs column (in DMM rows) ---
    if "obs" in sub.columns:
        dmm_obs = sub[sub["ref"] == "DMM"]
        if not dmm_obs.empty and dmm_obs["obs"].notna().any():
            exp_mean = dmm_obs.groupby("time")["obs"].mean()
            ax.plot(
                exp_mean.index, exp_mean.values,
                color=REF_PALETTE.get("obs", "#2ca02c"), marker="o", markersize=3,
                linewidth=LW_THICK, label=REF_LABELS.get("obs", "Experimental"), zorder=10,
            )

    # --- Sample reference (positive control) ---
    sample_ref = sub[sub["ref"] == "sample"]
    if not sample_ref.empty:
        sample_mean = sample_ref.groupby("time")["sim"].mean()
        sample_label = REF_DISPLAY_MAP.get("sample", REF_LABELS.get("sample", "Sample ref"))
        ax.plot(
            sample_mean.index, sample_mean.values,
            color=BARPLOT_REF_LINE_COLORS.get("positive control", REF_PALETTE.get("sample", "#98df8a")),
            linewidth=LW_MEDIUM, linestyle=BARPLOT_REF_LINE_STYLES.get("positive control", "--"),
            alpha=0.7, label=sample_label, zorder=7,
        )

    # --- DMM: individual random-seed trajectories + mean/variance ---
    dmm = sub[sub["ref"] == "DMM"]
    if not dmm.empty:
        # Individual seed trajectories (background)
        if show_seeds:
            seed_cols = [c for c in ["job", "features"] if c in dmm.columns]
            if seed_cols:
                first_seed = True
                for _, seed_df in dmm.groupby(seed_cols):
                    seed_ts = seed_df.sort_values("time")
                    ax.plot(
                        seed_ts["time"], seed_ts["sim"],
                        color=REF_PALETTE["DMM"], alpha=0.12,
                        linewidth=LW_THIN, zorder=3,
                        label=REF_LABELS["DMM"] + " (random seeds)" if first_seed else None,
                    )
                    first_seed = False

        # Mean and variance across seeds (foreground)
        dmm_stats = dmm.groupby("time")["sim"].agg(["mean", "std"]).reset_index()
        ax.fill_between(
            dmm_stats["time"],
            dmm_stats["mean"] - dmm_stats["std"],
            dmm_stats["mean"] + dmm_stats["std"],
            color=REF_PALETTE["DMM"], alpha=0.22, linewidth=0, zorder=4,
            label=REF_LABELS["DMM"] + " (±1 SD)",
        )
        ax.plot(
            dmm_stats["time"], dmm_stats["mean"],
            color=REF_PALETTE["DMM"], linewidth=LW_THICK,
            linestyle="-", label=REF_LABELS["DMM"] + " (mean)", zorder=6,
        )

    # --- Other references (avg_model as negative control, regressors) ---
    for ref_name in ["avg_model", "linreg", "lasso", "elasticnet"]:
        ref_sub = sub[sub["ref"] == ref_name]
        if ref_sub.empty:
            continue
        if refs is not None and ref_name not in refs:
            continue
        ref_mean = ref_sub.groupby("time")["sim"].mean()

        # Apply control styling for avg_model
        if ref_name == "avg_model":
            control_label = REF_DISPLAY_MAP.get("avg_model", REF_LABELS.get("avg_model", "Avg. model"))
            color = BARPLOT_REF_LINE_COLORS.get("negative control", REF_PALETTE.get("avg_model", "grey"))
            linestyle = BARPLOT_REF_LINE_STYLES.get("negative control", "--")
        else:
            control_label = REF_LABELS.get(ref_name, ref_name)
            color = REF_PALETTE.get(ref_name, "grey")
            linestyle = "--"

        ax.plot(
            ref_mean.index, ref_mean.values,
            color=color, linewidth=LW_MEDIUM, linestyle=linestyle,
            label=control_label, zorder=2,
        )

    # --- Labels ---
    obs_label = OBS_LABELS.get(observable, observable)
    cond_label = CONDITION_LABELS.get(condition, condition)
    ax.set_xlabel("Time (min)")
    ax.set_xticks([15, 30, 45, 60])
    ax.set_ylabel(f"{obs_label} [a.u.]")
    if title is None:
        cl_display = cell_line.replace("c", "", 1)
        title = f"{cl_display} — {cond_label}"
    ax.set_title(title)

    # --- Configure axis styling: keep left/bottom spines, show ticks ---
    configure_axis_spines(ax, top=False, right=False, left=True, bottom=True)
    ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
    ax.grid(True, axis="y", alpha=0.3)
    ax.grid(False, axis="x")

    return ax


## 1. Representative validation trajectories

Three validation-set trajectories selected at the Q10, Q50, and Q90
of the per-combo (cell line × condition × observable) RMSE distribution.
Each panel shows experimental data, DMM mean ± SD, and regression baselines.

In [ ]:
# ── Plot 3 trajectories + empty axis (left) + RMSE distribution (right) ───────────────
fig, axes = plt.subplots(1, 5, figsize=(9.5, 2), gridspec_kw={"width_ratios": [1.0, 1.6, 1, 1, 1]})
empty_ax = axes[0]
empty_ax.axis("off")
traj_axes = axes[2:6]
dist_ax = axes[1]

for ax, (tag, (cl, cond, obs)) in zip(traj_axes, rep_combos.items()):
    plot_trajectory(
        traj, cl, cond, obs,
        model_label=selected_model,
        context=selected_context,
        refs=["DMM", "elasticnet"],
        ax=ax,
        show_seeds=False,
    )
    obs_label = OBS_LABELS.get(obs, obs)
    cond_label = CONDITION_LABELS.get(cond, cond)
    cl_display = cl.replace("c", "", 1)
    rmse_val = combo_rmse[(cl, cond, obs)]
    ax.set_title(f"{cl_display} — {cond_label}")
    # Colour the quantile tag red, placed above the title
    ax.text(0.5, 0.95, f"{tag} ({rmse_val:.2f})", transform=ax.transAxes, ha="center", va="bottom",
            fontsize=FONTSIZE, color="#d62728")

# ── Consistent y-axis across all trajectory panels ─────────────────────
ymin = min(ax.get_ylim()[0] for ax in traj_axes)
ymax = max(ax.get_ylim()[1] for ax in traj_axes)
for ax in traj_axes:
    ax.set_ylim(ymin, ymax)

# Single legend (trajectory panels)
handles, labels = traj_axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(
    by_label.values(), by_label.keys(),
    loc="lower center", ncol=len(by_label),
    fontsize=FONTSIZE, frameon=False,
    bbox_to_anchor=(0.5, -0.08),
)
for ax in traj_axes:
    leg = ax.get_legend()
    if leg:
        leg.remove()

# ── RMSE distribution: DMM train/test, baselines on test, noise floor ──
plot_rmse_by_method(dist_ax, method_rmse, noise_sigma)
dist_ax.set_title("RMSE distribution")

# Mark the test quantiles of the representative trajectories on the DMM test box
test_rmse = method_rmse.query("ref == 'DMM' and dataset == 'val'")["rmse"]
for q in quantiles:
    dist_ax.hlines(
        test_rmse.quantile(q), 0.6, 1.4,
        color="#d62728", linestyle="--", alpha=0.7, linewidth=LW_THIN,
    )

fig.tight_layout()

fig.savefig(str(Path(_project_root) / "figures_paper" / "traj_representative_val.pdf"), bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
method_rmse.groupby(["ref", "dataset"])["rmse"].agg(["min", "median", "max"]).rename(columns={"min": "Min", "median": "Median", "max": "Max"})

In [ ]:
# t-test comparing train vs test RMSE distributions
from scipy.stats import ttest_ind
train_rmse = method_rmse.query("ref == 'DMM' and dataset == 'train'")["rmse"]
test_rmse = method_rmse.query("ref == 'DMM' and dataset == 'val'")["rmse"]
t_stat, p_value = ttest_ind(train_rmse, test_rmse, equal_var=False)
print(f"\nT-test comparing train vs test RMSE distributions:")
print(f"  t-statistic = {t_stat:.4f}, p-value = {p_value:.4e}")

## DMM vs. baselines: paired tests

Wilcoxon signed-rank tests over test-set (cell line × condition × observable)
combos, Holm-corrected across the five baselines. Combos of one cell line are
not independent; `p_cell_lines` repeats each comparison on one mean ΔRMSE per
cell line (exact sign-flip test).

In [ ]:
# Wilcoxon signed-rank tests, DMM vs. each baseline, over the test-set combos.
# delta = DMM - baseline, so negative values favour the DMM.
paired = paired_tests(method_rmse)
paired.to_csv("fig1c_paired_tests.csv", index=False)
display(paired)

for _, r in paired.iterrows():
    print(
        f"DMM vs {r.baseline}: n = {r.n_combos} combos ({r.n_cell_lines} cell lines), "
        f"median RMSE {r.median_rmse_dmm:.2f} vs {r.median_rmse_baseline:.2f} "
        f"({r.pct_lower:.0f}% lower), median ΔRMSE = {r.median_delta:.3f}, "
        f"p = {r.p:.2g} (Holm {r.p_holm:.2g})"
    )

# Sensitivity: the RMSEs as aggregate_and_log scored them, i.e. the regressors
# against replicate-averaged, gap-filled targets
display(
    paired_tests(
        baseline_comparison.combo_rmse(by_cl, selected_context, selected_features)
    )
)

## 2. Train vs test RMSE scatter

Scatter of median RMSE across cell lines: x = train, y = test.
Marker shape encodes the observable, colour encodes the condition.
Crosshairs show ± 1 SD across random seeds.
Points above the diagonal indicate a generalisation gap.

In [ ]:
# --- Train vs Test RMSE scatter (DMM only) ---
from matplotlib.lines import Line2D

dmm = by_cl[
    (by_cl["ref"] == "DMM") & (by_cl["context"] == selected_context)
].copy()

# Median and SE of RMSE per (observable, condition, dataset) across cell lines
stats = (
    dmm.groupby(["observable", "condition", "dataset"])["rmse"]
    .agg(["median", "std", "count"])
    .reset_index()
)
stats["se"] = stats["std"] / np.sqrt(stats["count"])

train_stats = stats[stats["dataset"] == "train"].rename(
    columns={"median": "median_train", "se": "se_train"}
).drop(columns=["dataset", "std", "count"])
val_stats = stats[stats["dataset"] == "val"].rename(
    columns={"median": "median_val", "se": "se_val"}
).drop(columns=["dataset", "std", "count"])

scatter_df = train_stats.merge(val_stats, on=["observable", "condition"])

# Marker shapes per observable
obs_markers = {
    "pERBB2_Y1248_obs": "o",
    "pERK_Y204_obs": "s",
    "pMEK_S222_obs": "D",
    "pRPS6KA1_S380_obs": "^",
}

# Condition colours
cond_colors = {
    "EGF": "#1f77b4",
    "iEGFR": "#ff7f0e",
    "iMEK": "#2ca02c",
}

fig, ax = plt.subplots(figsize=(6, 6))

for _, row in scatter_df.iterrows():
    obs = row["observable"]
    cond = row["condition"]
    x, y = row["median_train"], row["median_val"]
    xerr, yerr = row["se_train"], row["se_val"]
    ax.errorbar(
        x, y, xerr=xerr, yerr=yerr,
        fmt=obs_markers.get(obs, "o"),
        color=cond_colors.get(cond, "grey"),
        markersize=5, capsize=2, capthick=LW_THIN, elinewidth=LW_THIN,
        zorder=3,
    )

# Diagonal reference line
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(lims, lims, "k--", alpha=0.4, lw=LW_MEDIUM, zorder=1)
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("DMM median RMSE (train)")
ax.set_ylabel("DMM median RMSE (test)")
ax.set_title("Train vs validation RMSE by observable & condition\n(error bars = ±1 SE)")
ax.set_aspect("equal")

# Build legends
obs_handles = [
    Line2D([], [], marker=m, color="grey", linestyle="", markersize=5,
           label=OBS_LABELS.get(o, o))
    for o, m in obs_markers.items()
]
cond_handles = [
    Line2D([], [], marker="o", color=c, linestyle="", markersize=5,
           label=CONDITION_LABELS.get(c_key, c_key))
    for c_key, c in cond_colors.items()
]

leg1 = ax.legend(handles=obs_handles, title="Observable", loc="upper left",
                 fontsize=FONTSIZE, title_fontsize=FONTSIZE)
ax.add_artist(leg1)
configure_axis_spines(ax, top=False, right=False, left=True, bottom=True)
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
ax.grid(False)
ax.legend(handles=cond_handles, title="Condition", loc="lower right",
          fontsize=FONTSIZE, title_fontsize=FONTSIZE)

plt.tight_layout()
plt.show()

In [ ]:

# ── Unpaired test: train RMSE ≠ test RMSE per (obs, cond) combo ──────
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

dmm_test = by_cl[
    (by_cl["ref"] == "DMM") & (by_cl["context"] == selected_context)
].copy()

# Per cell-line median RMSE (across seeds) for each (obs, cond, dataset)
cl_rmse = (
    dmm_test.groupby(["observable", "condition", "dataset", "cell_line"])["rmse"]
    .median()
    .reset_index()
)

results = []
for (obs, cond), grp in cl_rmse.groupby(["observable", "condition"]):
    train = grp.loc[grp["dataset"] == "train", "rmse"].values
    val   = grp.loc[grp["dataset"] == "val",   "rmse"].values
    u_stat, p_val = mannwhitneyu(val, train, alternative="greater")
    results.append({
        "observable": OBS_LABELS.get(obs, obs),
        "condition":  CONDITION_LABELS.get(cond, cond),
        "n_train": len(train),
        "n_val":   len(val),
        "median_train": np.median(train),
        "median_val":   np.median(val),
        "U": u_stat,
        "p_raw": p_val,
    })

results_df = pd.DataFrame(results)

# Benjamini-Hochberg FDR correction
reject, p_adj, _, _ = multipletests(results_df["p_raw"], method="fdr_bh")
results_df["p_adj"] = p_adj
results_df["significant"] = reject

# Display
print("Mann–Whitney U test per (observable, condition) — H₁: val RMSE > train RMSE")
print("Multiple-testing correction: Benjamini–Hochberg FDR\n")
for _, r in results_df.iterrows():
    sig = "✓" if r["significant"] else "✗"
    print(
        f"  {r['observable']:8s} / {r['condition']:14s}  "
        f"n=({r['n_train']},{r['n_val']})  "
        f"med train={r['median_train']:.4f}  med val={r['median_val']:.4f}  "
        f"U={r['U']:.0f}  p_raw={r['p_raw']:.4g}  p_adj={r['p_adj']:.4g}  {sig}"
    )

n_sig = results_df["significant"].sum()


In [ ]:
display(results_df)